# Persona-language translation — Phase 1: direct-source decoder baseline

Plan v2 (`encoder_decoder_bridge_e2e_plan_v2.md`), Phase 1; implementation notes in `stitched-e2e-plan.md`.

**Task.** Translate an English source sentence into natural Roman-script Hinglish **in one of six persona voices**, preserving its meaning: wording, rhythm and framing may change; facts, intent and the proposition may not. Data: v3, 60 English sources × 6 personas = 360 pairs, split by source into train 40 sources (240 pairs), validation 10 (60), blind test 10 (60).

**What this notebook is.** The decoder sees the source sentence directly in its prompt, plus the persona's name and description, and is fine-tuned with LoRA. It is the *direct-source baseline*: it selects the decoder for the bridge, and it does **not** show that an encoder–bridge path works (that needs the causal tests of Phase 4).

**This notebook runs `qwen-instruct+hinglish`.** The three runs are three copies of the same notebook (`persona-translation-baseline-<decoder>.ipynb`), identical except for `DECODER` in Cell 0, so each keeps its own outputs:

| `DECODER` | Model | Role |
|---|---|---|
| `gemma-4b` | `google/gemma-3-4b-it`, 4-bit NF4, fp32 compute (QLoRA) | compare with the persona pilots |
| `qwen-instruct` | `Qwen/Qwen2.5-1.5B-Instruct`, fp32 | clean Qwen baseline; preferred bridge candidate |
| `qwen-instruct+hinglish` | the same, with `sarthaksenapati/qwen1.5b-hinglish-sft-v2` merged in | **diagnostic only:** the adapter was trained on the *base* Qwen2.5-1.5B, so merging it into Instruct is not an equivalent adaptation |

Everything else is identical across the three runs: data, prompt, LoRA settings, checkpoint selection, decoding and evaluation.

**Protocol**
- Train on 240 pairs; **choose the checkpoint by validation loss** (60 pairs); early stopping after two epochs without improvement; per-epoch probe outputs from two validation sources.
- **Test (60) is generated once, after selection, and only reported** (Cell 9); it is never used for a decision.
- Decoding: greedy, 80 new tokens at most, **no repetition guard**, so repetition is measured, not suppressed.

**Phase 1 gates** (for choosing the decoder; LoRA outputs):

| Metric | Gate | Where |
|---|---|---|
| Correct stopping | ≥ 95 % | Cell 7 (automatic) |
| Repetition failures (a word 4-gram occurring twice) | ≤ 5 % | Cell 7 (automatic) |
| Meaning preservation vs the English source | mean ≥ 4 / 5 | Cell 8 sheet, 30 fixed validation pairs (human) |
| Fluency; unsupported additions | reported | Cell 8 sheet (human) |
| Persona recognition, blind 6-way | reported; 50 % stretch (chance 17 %) | Cell 8 blind sheet (human) |

chrF against the reference, untranslated English, and persona overlap are reported as diagnostics. The decoder is chosen on validation results, stopping reliability, translation quality and resource use, never on training loss alone.

In [18]:
# Cell 0 — Choose the decoder for this run; Hugging Face token for Gemma only
# Clear the token before saving or sharing the notebook, so it never ends up in a file.

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # run on the 2nd T4 (Gemma uses the 1st); must run before torch is imported,
                                           # i.e. in a fresh kernel. Use "0" when this notebook has the machine to itself.

DECODER = "qwen-instruct+hinglish"  # this notebook's run; the other two runs have their own notebooks
HF_TOKEN = ""                 # not needed for Qwen (public); leave empty

## Cell 1 — Setup, pinned revisions, adapter check

`bitsandbytes` is installed for all runs (only Gemma uses it). No other packages: chrF is computed in the notebook (Cell 7). Each repository is pinned to the revision found now. For the diagnostic adapter run, the adapter's configured base model is printed next to the model it is merged into; the mismatch is expected and recorded, not an error.

In [19]:
# Cell 1 — Setup: packages, decoder settings, versions, pinned revisions

!pip uninstall -y -q torchao                 # Kaggle ships torchao 0.10; peft refuses versions below 0.16
!pip install -q "transformers==5.17.0" "peft==0.21.2" -U bitsandbytes

import json, random, re, statistics, time, urllib.request
from collections import Counter
from itertools import combinations

import peft
import torch
import transformers
from huggingface_hub import HfApi, login

DECODERS = {
    "gemma-4b": {"model": "google/gemma-3-4b-it", "adapter": None},
    "qwen-instruct": {"model": "Qwen/Qwen2.5-1.5B-Instruct", "adapter": None},
    "qwen-instruct+hinglish": {"model": "Qwen/Qwen2.5-1.5B-Instruct",
                               "adapter": "sarthaksenapati/qwen1.5b-hinglish-sft-v2"},
}
CFG = DECODERS[DECODER]
IS_GEMMA = DECODER.startswith("gemma")
DATA = ("https://raw.githubusercontent.com/skamalj/llms-the-hard-way/hindi-decoder-compare/"
        "project4-hindi-decoder/data/persona_translation_v3/")
MAX_NEW_TOKENS = 80
SEED = 42
DTYPE = torch.float32
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

HF_TOKEN = HF_TOKEN.strip() or None
if HF_TOKEN:
    login(HF_TOKEN)
assert HF_TOKEN or not IS_GEMMA, "Gemma 3 is gated: paste a Hugging Face token in Cell 0"

api = HfApi()
REV = {r: api.model_info(r, token=HF_TOKEN).sha for r in (CFG["model"], CFG["adapter"]) if r}
ADAPTER_NOTE = None
if CFG["adapter"]:
    adapter_cfg = json.load(open(api.hf_hub_download(CFG["adapter"], "adapter_config.json",
                                                     revision=REV[CFG["adapter"]])))
    ADAPTER_NOTE = {"trained_on": adapter_cfg["base_model_name_or_path"], "merged_into": CFG["model"],
                    "r": adapter_cfg.get("r"), "alpha": adapter_cfg.get("lora_alpha")}
    if ADAPTER_NOTE["trained_on"] != CFG["model"]:
        print(f"NOTE (expected, diagnostic run): the adapter was trained on {ADAPTER_NOTE['trained_on']}, "
              f"and is merged into {CFG['model']}")

VERSIONS = {
    "decoder": DECODER,
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "peft": peft.__version__,
    "device": torch.cuda.get_device_name(DEVICE) if DEVICE.type == "cuda" else "cpu",
    "revisions": REV,
    "adapter": ADAPTER_NOTE,
}
print(json.dumps(VERSIONS, indent=1))

adapter_config.json:   0%|          | 0.00/1.10k [00:00<?, ?B/s]

NOTE (expected, diagnostic run): the adapter was trained on Qwen/Qwen2.5-1.5B, and is merged into Qwen/Qwen2.5-1.5B-Instruct
{
 "decoder": "qwen-instruct+hinglish",
 "torch": "2.11.0+cu128",
 "transformers": "5.16.1",
 "peft": "0.20.0",
 "device": "Tesla T4",
 "revisions": {
  "Qwen/Qwen2.5-1.5B-Instruct": "989aa7980e4cf806f80c7fef2b1adb7bc71aa306",
  "sarthaksenapati/qwen1.5b-hinglish-sft-v2": "942fd0b2ebd2119d772f18952de6f8d372a6448b"
 },
 "adapter": {
  "trained_on": "Qwen/Qwen2.5-1.5B",
  "merged_into": "Qwen/Qwen2.5-1.5B-Instruct",
  "r": 32,
  "alpha": 64
 }
}


## Cell 2 — Data v3, the prompt, and the fixed samples

The three split files and the manifest (persona names and descriptions) come from the repository. The prompt is v3's own instruction with **one added sentence, the persona's description** (plan v2: a name alone is not enough):

```
Translate the English sentence into natural Roman-script Hinglish in the voice of Don. Persona: <description>
Preserve the original meaning; do not add new events or facts.

English: <source>
```

The cell checks that removing the added sentence gives back v3's `input` exactly, and that no source appears in two splits.

- **Fixed human-scoring sample:** every other validation source (5 sources × 6 personas = 30 pairs), the same for all three decoders.
- **Probe sources** (printed every epoch): the two validation sources outside that sample.
- **Out-of-domain sources** (no references): the requirements' two everyday sentences and four more. They were written by the notebook author, so they are a diagnostic, not an independent test.

In [20]:
# Cell 2 — Load v3, build the prompts, check the splits, fix the samples

def fetch(name):
    return json.load(urllib.request.urlopen(DATA + name))


MANIFEST = fetch("persona_translation_dataset_v3.json")
SPLITS = {s: fetch(f"{s}_v3.json") for s in ("train", "validation", "test")}
PERSONAS = list(MANIFEST["personas"])
NAME = {p: MANIFEST["personas"][p]["display_name"] for p in PERSONAS}
DESC = {p: MANIFEST["personas"][p]["description"] for p in PERSONAS}

HEAD = "Translate the English sentence into natural Roman-script Hinglish in the voice of {name}."
TAIL = " Preserve the original meaning; do not add new events or facts.\n\nEnglish: {source}"


def make_input(persona, source, with_description=True):
    description = f" Persona: {DESC[persona]}" if with_description else ""
    return HEAD.format(name=NAME[persona]) + description + TAIL.format(source=source)


for split, rows in SPLITS.items():
    for ex in rows:
        assert make_input(ex["persona_id"], ex["source"], with_description=False) == ex["input"], ex["id"]
        ex["kind"] = split
        ex["input"] = make_input(ex["persona_id"], ex["source"])

TRAIN, VAL, TEST = SPLITS["train"], SPLITS["validation"], SPLITS["test"]
SOURCES = {k: {ex["source_id"] for ex in rows} for k, rows in SPLITS.items()}
assert not (SOURCES["train"] & SOURCES["validation"] or SOURCES["train"] & SOURCES["test"]
            or SOURCES["validation"] & SOURCES["test"])

OOD_SOURCES = [
    "I am going to eat fruit today.",
    "My Wi-Fi is disconnected.",
    "Please call me when you reach home.",
    "The meeting has been moved to Monday morning.",
    "I lost my keys again.",
    "Thank you for helping me move the furniture.",
]
OOD = [{"id": f"ood{i}_{p}", "source_id": f"ood{i}", "kind": "ood", "persona_id": p, "source": s,
        "target": None, "input": make_input(p, s)}
       for i, s in enumerate(OOD_SOURCES, 1) for p in PERSONAS]

VAL_SOURCES = sorted(SOURCES["validation"])
SAMPLE_SOURCES = VAL_SOURCES[::2]
PROBE_SOURCES = VAL_SOURCES[1::2][:2]

print(f"personas {PERSONAS}")
print({k: len(v) for k, v in SPLITS.items()}, "| out-of-domain", len(OOD))
print(f"human-scoring sample {SAMPLE_SOURCES} ({len(SAMPLE_SOURCES) * 6} pairs) | probes {PROBE_SOURCES}")
print("--- example input:\n" + TRAIN[0]["input"])
print("--- its target:\n" + TRAIN[0]["target"])

personas ['vijay_deewar', 'don', 'shahenshah', 'anthony_gonsalves', 'vijay_agneepath', 'jai_sholay']
{'train': 240, 'validation': 60, 'test': 60} | out-of-domain 36
human-scoring sample ['s041', 's043', 's045', 's047', 's049'] (30 pairs) | probes ['s042', 's044']
--- example input:
Translate the English sentence into natural Roman-script Hinglish in the voice of Vijay (Deewar). Persona: Proud, self-made, quietly defiant; carries the weight of injustice and refuses to be pushed around. Preserve the original meaning; do not add new events or facts.

English: I don't trust someone who changes their story every time.
--- its target:
Jo aadmi har baar apni baat badal de, us par bharosa karna mere bas ki baat nahi.


## Cell 3 — Load the decoder, check the stop token, build the training tensors

- **Gemma:** 4-bit NF4 weights with fp32 compute on one GPU (`Gemma3ForConditionalGeneration`, text only). **Qwen:** fp32; for the diagnostic run, the Hinglish adapter is applied and merged into the weights (`merge_and_unload`).
- **Stop-token check.** The model's end-of-turn token must have a trained embedding, or LoRA (which does not train embeddings) cannot teach the model to emit it. This is what broke Qwen2.5 *base* (0/84 stops). The cell prints the embedding norm of each stop token against the median token.
- Each example is the model's own chat template: user turn = the prompt, assistant turn = the v3 target. The answer part (target + the template's closing tokens) is taken from the template's own rendering; labels are `-100` on the prompt, so the loss covers the translation **and** the stop.
- `generate`: greedy, 80 new tokens at most, no repetition guard; records how each output ended.

In [21]:
# Cell 3 — Load the decoder, stop-token check, template, tensors, generate

from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, Gemma3ForConditionalGeneration

tok = AutoTokenizer.from_pretrained(CFG["model"], revision=REV[CFG["model"]], token=HF_TOKEN)
if IS_GEMMA:
    quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=DTYPE)
    base = Gemma3ForConditionalGeneration.from_pretrained(CFG["model"], revision=REV[CFG["model"]], dtype=DTYPE,
                                                          token=HF_TOKEN, quantization_config=quant,
                                                          device_map={"": 0}, attn_implementation="eager")
    eos = base.generation_config.eos_token_id
    STOPS = sorted(set(eos if isinstance(eos, list) else [eos]))
else:
    base = AutoModelForCausalLM.from_pretrained(CFG["model"], revision=REV[CFG["model"]], dtype=DTYPE).to(DEVICE)
    if CFG["adapter"]:
        base = PeftModel.from_pretrained(base, CFG["adapter"], revision=REV[CFG["adapter"]]).merge_and_unload()
        assert not any("lora_" in n for n, _ in base.named_parameters()), "the Hinglish adapter is not merged"
    STOPS = sorted({tok.convert_tokens_to_ids("<|im_end|>"), tok.convert_tokens_to_ids("<|endoftext|>")})
PAD = tok.pad_token_id if tok.pad_token_id is not None else STOPS[0]
print(f"GPU memory after loading: {torch.cuda.memory_allocated(DEVICE) / 2**30:.1f} GB")

norms = base.get_input_embeddings().weight.detach().float().norm(dim=1)
print(f"stop-token embeddings (median token norm {norms.median().item():.3f}):")
for t in STOPS:
    print(f"  {tok.convert_ids_to_tokens(t):16s} id {t:6d}  norm {norms[t].item():.3f}")


def prompt_text(user):
    return tok.apply_chat_template([{"role": "user", "content": user}], add_generation_prompt=True, tokenize=False)


def encode(ex):
    """Prompt tokens + answer tokens, tokenized apart (exactly what generation sees). The answer is the template's
    own rendering after the prompt: the target plus the closing end-of-turn tokens."""
    msgs = [{"role": "user", "content": ex["input"]}, {"role": "assistant", "content": ex["target"]}]
    full, head = tok.apply_chat_template(msgs, tokenize=False), prompt_text(ex["input"])
    assert full.startswith(head), "the generation prompt is not a prefix of the full conversation"
    answer = full[len(head):]
    assert answer.startswith(ex["target"]) and len(answer) > len(ex["target"]), repr(answer)
    prompt = tok(head, add_special_tokens=False)["input_ids"]
    reply = tok(answer, add_special_tokens=False)["input_ids"]
    assert any(t in STOPS for t in reply), "the answer carries no stop token"
    return torch.tensor(prompt + reply), torch.tensor([-100] * len(prompt) + reply)


def collate(examples):
    pairs = [encode(ex) for ex in examples]
    T = max(len(ids) for ids, _ in pairs)
    ids = torch.full((len(pairs), T), PAD)
    labels = torch.full((len(pairs), T), -100)
    mask = torch.zeros((len(pairs), T), dtype=torch.long)
    for i, (x, y) in enumerate(pairs):
        ids[i, :len(x)], labels[i, :len(y)], mask[i, :len(x)] = x, y, 1
    batch = {"input_ids": ids.to(DEVICE), "attention_mask": mask.to(DEVICE), "labels": labels.to(DEVICE)}
    if IS_GEMMA:
        batch["token_type_ids"] = torch.zeros_like(ids).to(DEVICE)      # 0 = text token (no images)
    return batch


DEVA = re.compile(r"[ऀ-ॿ]")
LATN = re.compile(r"[A-Za-z]")


@torch.no_grad()
def generate(user):
    ids = tok(prompt_text(user), return_tensors="pt", add_special_tokens=False).to(DEVICE)
    out = model.generate(**ids, do_sample=False, num_beams=1, max_new_tokens=MAX_NEW_TOKENS, eos_token_id=STOPS,
                         pad_token_id=PAD, temperature=None, top_p=None, top_k=None)
    new = out[0, ids["input_ids"].shape[1]:].tolist()
    text = tok.decode(new, skip_special_tokens=True).strip()
    d, l = len(DEVA.findall(text)), len(LATN.findall(text))
    ended = "stop token" if new and new[-1] in STOPS else "token limit" if len(new) >= MAX_NEW_TOKENS else "other"
    return {"output": text, "new_tokens": len(new), "ended": ended,
            "devanagari_share": round(d / (d + l), 3) if d + l else None}


ids, labels = encode(TRAIN[0])
print("--- one training example as the model sees it:")
print(tok.decode(ids))
print("--- the part that carries the loss:")
print(repr(tok.decode(ids[labels != -100])))

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

adapter_model.safetensors: reconstructing file:   0%|          |  0.00B /  148MB            

adapter_model.safetensors: downloading bytes:           |  0.00B            

GPU memory after loading: 10.7 GB
stop-token embeddings (median token norm 1.029):
  <|endoftext|>    id 151643  norm 1.134
  <|im_end|>       id 151645  norm 0.422
--- one training example as the model sees it:
<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
Translate the English sentence into natural Roman-script Hinglish in the voice of Vijay (Deewar). Persona: Proud, self-made, quietly defiant; carries the weight of injustice and refuses to be pushed around. Preserve the original meaning; do not add new events or facts.

English: I don't trust someone who changes their story every time.<|im_end|>
<|im_start|>assistant
Jo aadmi har baar apni baat badal de, us par bharosa karna mere bas ki baat nahi.<|im_end|>

--- the part that carries the loss:
'Jo aadmi har baar apni baat badal de, us par bharosa karna mere bas ki baat nahi.<|im_end|>\n'


## Cell 4 — LoRA training, checkpoint chosen by validation loss

- LoRA rank 16, alpha 32, dropout 0.05, on the attention and MLP projections (for Gemma, only the language model's, by pattern). Base weights frozen.
- Gemma: `prepare_model_for_kbit_training` with gradient checkpointing. Qwen: plain fp32.
- Micro-batches of 2 with 2-step accumulation (effective batch 4); AdamW, learning rate 1e-4, linear decay over the 8-epoch maximum.
- After every epoch: training loss, **validation loss** (60 pairs), and the probe outputs (two validation sources × six personas). Early stopping after two epochs without a lower validation loss; the best validation epoch is kept. Epoch 0 is the decoder without LoRA.

In [22]:
# Cell 4 — Wrap with LoRA, train with early stopping on validation loss, print probes each epoch

from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MAX_EPOCHS, PATIENCE, MICRO, ACCUM, LR = 8, 2, 2, 2, 1e-4
PROJ = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
TARGETS = rf".*language_model.*\.({'|'.join(PROJ)})" if IS_GEMMA else PROJ
LORA = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, task_type="CAUSAL_LM", target_modules=TARGETS)

if IS_GEMMA:
    base = prepare_model_for_kbit_training(base, use_gradient_checkpointing=True,
                                           gradient_checkpointing_kwargs={"use_reentrant": False})
torch.manual_seed(SEED)
model = get_peft_model(base, LORA)
model.print_trainable_parameters()
assert not any("vision" in n for n, p in model.named_parameters() if p.requires_grad)

trainable = [p for p in model.parameters() if p.requires_grad]
opt = torch.optim.AdamW(trainable, lr=LR, weight_decay=0.0)
steps = MAX_EPOCHS * -(-len(TRAIN) // (MICRO * ACCUM))
sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: 1 - s / steps)


@torch.no_grad()
def validation_loss(batch=4):
    model.eval()
    total = 0.0
    for i in range(0, len(VAL), batch):
        chunk = VAL[i:i + batch]
        total += model(**collate(chunk)).loss.item() * len(chunk)
    return total / len(VAL)


def lora_state():
    return {k: v.detach().clone() for k, v in model.state_dict().items() if "lora_" in k}


def probe(epoch):
    model.eval()
    rows = [{"epoch": epoch, **ex, **generate(ex["input"])} for ex in VAL if ex["source_id"] in PROBE_SOURCES]
    for sid in PROBE_SOURCES:
        print(f"  probe | {sid}: {next(r['source'] for r in rows if r['source_id'] == sid)}")
        for r in rows:
            if r["source_id"] == sid:
                print(f"    {r['persona_id']:18s} {r['output']}" + ("" if r["ended"] == "stop token" else "  [NO STOP]"))
    return rows


def train_epoch(order):
    model.train()
    losses = []
    batches = [order[i:i + MICRO] for i in range(0, len(order), MICRO)]
    for n, batch in enumerate(batches, 1):
        loss = model(**collate(batch)).loss
        (loss / ACCUM).backward()
        losses.append(loss.item())
        if n % ACCUM == 0 or n == len(batches):
            opt.step()
            sched.step()
            opt.zero_grad()
    return sum(losses) / len(losses)


LOG = [{"epoch": 0, "train_loss": None, "validation_loss": round(validation_loss(), 4)}]
best = (LOG[0]["validation_loss"], 0, lora_state())
PROBES, since_best = [], 0
rng = random.Random(SEED)
t0 = time.perf_counter()
for epoch in range(1, MAX_EPOCHS + 1):
    train_loss = train_epoch(rng.sample(TRAIN, len(TRAIN)))
    LOG.append({"epoch": epoch, "train_loss": round(train_loss, 4), "validation_loss": round(validation_loss(), 4)})
    print(LOG[-1], f"| {time.perf_counter() - t0:.0f}s")
    PROBES += probe(epoch)
    if LOG[-1]["validation_loss"] < best[0]:
        best, since_best = (LOG[-1]["validation_loss"], epoch, lora_state()), 0
    else:
        since_best += 1
    if since_best >= PATIENCE:
        print(f"early stop: no lower validation loss for {PATIENCE} epochs")
        break

model.load_state_dict(best[2], strict=False)
BEST_EPOCH = best[1]
TRAIN_SECONDS = round(time.perf_counter() - t0)
print(f"kept epoch {BEST_EPOCH} (validation loss {best[0]}); epoch 0, no LoRA: {LOG[0]['validation_loss']}")

trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820
{'epoch': 1, 'train_loss': 3.1808, 'validation_loss': 2.9404} | 104s
  probe | s042: I will support the decision once everyone understands the trade-offs.
    vijay_deewar       Aaj ka decision ko samjho. Phir uska khatam karne mein baat karo.
    don                Decision ko koi batao. Phir decide karoge.
    shahenshah         Aaj ka jagah par sab kuch samjha kar batao. Phir main decision ko abhi se support karo.
    anthony_gonsalves  Team meeting ke baad, sabko chhodkar pehle se batao ki boss ka decision ko koi nahi hoga.
    vijay_agneepath    Decision ko samjho. Phir baad mein koi batao.
    jai_sholay         Decision ko samjho. Phir koi batao ki main abhi nahi madad karunga.
  probe | s044: I want a solution that works for the people using it, not just the people building it.
    vijay_deewar       Aajki koi bata hai? Aapka baat nahi.
    don                Solution ko users ke liye design karo. D

## Cell 5 — Generate

- **Without LoRA** (the decoder as loaded, adapter switched off): validation and out-of-domain sources, as the reference point.
- **With LoRA** (the kept checkpoint): validation, out-of-domain, and the **test** split. Test outputs are only summarised in Cell 9.

In [23]:
# Cell 5 — Greedy generation without and with LoRA

model.eval()
RESULTS = []
for variant, rows in [("no lora", VAL + OOD), ("lora", VAL + OOD + TEST)]:
    t0 = time.perf_counter()
    for ex in rows:
        if variant == "no lora":
            with model.disable_adapter():
                r = generate(ex["input"])
        else:
            r = generate(ex["input"])
        RESULTS.append({"variant": variant, "id": ex["id"], "kind": ex["kind"], "source_id": ex["source_id"],
                        "persona": ex["persona_id"], "source": ex["source"], "reference": ex["target"], **r})
    print(f"{variant}: {len(rows)} outputs in {time.perf_counter() - t0:.0f}s "
          f"({(time.perf_counter() - t0) / len(rows):.1f}s each)")

no lora: 96 outputs in 190s (2.0s each)
lora: 156 outputs in 302s (1.9s each)


## Cell 6 — Validation outputs, side by side

For each validation source: the English source, then per persona the output without LoRA, with LoRA, and the v3 reference. Then the out-of-domain outputs (with LoRA). Outputs that did not end on a stop token are marked.

In [24]:
# Cell 6 — Validation and out-of-domain outputs

BY_ID = {(r["variant"], r["id"]): r for r in RESULTS}


def show(r):
    return r["output"] + ("" if r["ended"] == "stop token" else f"   [{r['ended'].upper()}]")


for sid in VAL_SOURCES + [f"ood{i}" for i in range(1, len(OOD_SOURCES) + 1)]:
    rows = [ex for ex in VAL + OOD if ex["source_id"] == sid]
    tag = "sample" if sid in SAMPLE_SOURCES else "probe" if sid in PROBE_SOURCES else rows[0]["kind"]
    print(f"\n=== {sid} [{tag}] English: {rows[0]['source']}")
    for ex in rows:
        print(f"  {ex['persona_id']}")
        print(f"    {'no lora':9s} | {show(BY_ID[('no lora', ex['id'])])}")
        print(f"    {'lora':9s} | {show(BY_ID[('lora', ex['id'])])}")
        if ex["target"]:
            print(f"    {'reference':9s} | {ex['target']}")


=== s041 [sample] English: We should celebrate the progress without ignoring the remaining work.
  vijay_deewar
    no lora   | Hinglish version with Vijay's persona:

"Shuruat aur baad ka progress ko celebrate karo, lekin sabse pehle remaining work ko bhi dhyan mein rakho. Iska tareeka Deewar ki swarganand se."
    lora      | Aaj bharosa hai, lekin khatam nahi. Phir kaam hoga.
    reference | Jitna aage aaye hain uski khushi mana sakte hain, par baaki kaam bhoolenge nahi.
  don
    no lora   | Don ka bhi hinglish hai - "Progress ko celebrate karo, lekin sabse zaroori baat, remaining work ko dhyan mein rakho." Aapko don ki cool aur calculated persona chahiye, woh naturally translate kar sakta hai.
    lora      | Progress ka bharosa hai. Remaining work ko ek baar check karo. Two steps ahead.
    reference | Progress acknowledge karo. Remaining work ko invisible mat banao.
  shahenshah
    no lora   | Shahenshah ki swadon: "Tumhare tarike mein progress kyun bhi na bhoolo, lekin baad m

## Cell 7 — Automatic checks and the two automatic gates

Per variant and split:

- **Stopping:** ended on a stop token before the 80-token limit.
- **Repetition failure:** any word 4-gram occurs twice in the output.
- **Devanagari:** any Devanagari character (the target is Roman script).
- **chrF** against the v3 reference (character n-gram F-score, computed in the notebook: n-grams 1–6, spaces removed, beta 2; Popović 2015): a diagnostic of closeness to *one* acceptable translation, not a meaning score.
- **Untranslated English:** share of the output's words that are copied from the English source; outputs above 60 % are counted as largely untranslated (some English words are normal Hinglish code-switching).
- **Persona overlap:** mean word overlap (Jaccard) between the six personas' outputs for the same source; yardstick: the v3 references themselves.

The two automatic gates (stopping ≥ 95 %, repetition ≤ 5 %) are computed on the LoRA outputs for validation + out-of-domain.

In [25]:
# Cell 7 — Signals, chrF, untranslated English, persona overlap, automatic gates

def corpus_chrf(hyps, refs, max_n=6, beta=2.0):
    """chrF (Popovic 2015) at corpus level: character n-grams 1..6 with spaces removed, n-gram counts summed over
    the corpus, precision and recall averaged over n, then the F-score with beta 2, on a 0-100 scale."""
    precisions, recalls = [], []
    for n in range(1, max_n + 1):
        match = hyp_total = ref_total = 0
        for h, r in zip(hyps, refs):
            h, r = h.replace(" ", ""), r.replace(" ", "")
            hc = Counter(h[i:i + n] for i in range(len(h) - n + 1))
            rc = Counter(r[i:i + n] for i in range(len(r) - n + 1))
            match += sum((hc & rc).values())
            hyp_total += sum(hc.values())
            ref_total += sum(rc.values())
        if hyp_total and ref_total:
            precisions.append(match / hyp_total)
            recalls.append(match / ref_total)
    if not precisions:
        return 0.0
    p, r = statistics.mean(precisions), statistics.mean(recalls)
    return 100 * (1 + beta**2) * p * r / (beta**2 * p + r) if p + r else 0.0


def words(t):
    return re.findall(r"[a-zऀ-ॿ']+", t.lower())


def repetition_failure(t):
    w = words(t)
    grams = [" ".join(w[i:i + 4]) for i in range(len(w) - 3)]
    return len(grams) != len(set(grams))


def copied_english(r):
    out, src = words(r["output"]), set(words(r["source"]))
    return sum(x in src for x in out) / len(out) if out else 0.0


def jaccard(a, b):
    a, b = set(words(a)), set(words(b))
    return len(a & b) / len(a | b) if a | b else 1.0


def persona_overlap(lines):
    return statistics.mean(jaccard(a, b) for a, b in combinations(lines, 2))


def select(variant, kinds):
    return [r for r in RESULTS if r["variant"] == variant and r["kind"] in kinds]


def summary(rs):
    n = len(rs)
    s = {"n": n,
         "stop": sum(r["ended"] == "stop token" for r in rs) / n,
         "repetition": sum(repetition_failure(r["output"]) for r in rs) / n,
         "devanagari": sum((r["devanagari_share"] or 0) > 0 for r in rs) / n,
         "untranslated": sum(copied_english(r) > 0.6 for r in rs) / n,
         "median_tokens": statistics.median(r["new_tokens"] for r in rs)}
    refs = [r for r in rs if r["reference"]]
    if refs:
        s["chrf"] = corpus_chrf([r["output"] for r in refs], [r["reference"] for r in refs])
    groups = {}
    for r in rs:
        groups.setdefault(r["source_id"], []).append(r["output"])
    s["persona_overlap"] = statistics.mean(persona_overlap(v) for v in groups.values() if len(v) == 6)
    return s


ref_groups = {}
for ex in VAL + TEST + TRAIN:
    ref_groups.setdefault(ex["source_id"], []).append(ex["target"])
YARDSTICK = statistics.mean(persona_overlap(v) for v in ref_groups.values())
print(f"yardstick: persona overlap of the v3 references {YARDSTICK:.3f}\n")

SUMMARY = {}
print(f"{'variant':8s} {'split':10s} {'n':>3s} {'stop':>6s} {'repet.':>7s} {'deva':>6s} {'untransl':>9s} "
      f"{'chrF':>6s} {'overlap':>8s} {'med.tok':>8s}")
for variant in ("no lora", "lora"):
    for kind in ("validation", "ood"):
        s = SUMMARY[(variant, kind)] = summary(select(variant, {kind}))
        chrf = f"{s['chrf']:.1f}" if "chrf" in s else "—"
        print(f"{variant:8s} {kind:10s} {s['n']:3d} {s['stop']:6.0%} {s['repetition']:7.0%} {s['devanagari']:6.0%} "
              f"{s['untranslated']:9.0%} {chrf:>6s} {s['persona_overlap']:8.3f} {s['median_tokens']:8.0f}")

print("\nLoRA chrF per persona (validation):")
for p in PERSONAS:
    rs = [r for r in select("lora", {"validation"}) if r["persona"] == p]
    print(f"  {p:18s} {corpus_chrf([r['output'] for r in rs], [r['reference'] for r in rs]):5.1f}")

gate = summary(select("lora", {"validation", "ood"}))
GATES = {"stopping >= 95%": gate["stop"] >= 0.95, "repetition <= 5%": gate["repetition"] <= 0.05}
print(f"\nAUTOMATIC GATES (LoRA, validation + out-of-domain, n={gate['n']})")
print(f"  stopping   {gate['stop']:.1%} (gate >= 95%): {'PASS' if GATES['stopping >= 95%'] else 'FAIL'}")
print(f"  repetition {gate['repetition']:.1%} (gate <= 5%): {'PASS' if GATES['repetition <= 5%'] else 'FAIL'}")
print("  meaning preservation, fluency, additions, persona recognition: human, Cell 8")

yardstick: persona overlap of the v3 references 0.095

variant  split        n   stop  repet.   deva  untransl   chrF  overlap  med.tok
no lora  validation  60    97%      0%     0%        0%   22.8    0.204       50
no lora  ood         36    97%      0%     0%        0%      —    0.201       40
lora     validation  60   100%      0%     0%        0%   22.8    0.162       28
lora     ood         36   100%      0%     0%        0%      —    0.169       25

LoRA chrF per persona (validation):
  vijay_deewar        19.0
  don                 31.0
  shahenshah          21.9
  anthony_gonsalves   23.8
  vijay_agneepath     20.9
  jai_sholay          19.3

AUTOMATIC GATES (LoRA, validation + out-of-domain, n=96)
  stopping   100.0% (gate >= 95%): PASS
  repetition 0.0% (gate <= 5%): PASS
  meaning preservation, fluency, additions, persona recognition: human, Cell 8


## Cell 8 — Human-scoring sheets (fixed sample: 30 validation pairs, LoRA outputs)

**Sheet A — translation quality.** For each pair: the English source, the persona and the output (the reference is deliberately not shown; meaning is judged against the *source*). Score:
- **meaning** 1–5 against the English source (5 = the proposition, facts and intent all preserved);
- **fluency** 1–5 (natural conversational Roman-script Hinglish);
- **additions:** none / harmless (comic or persona framing) / material (an invented fact or event, or changed meaning).

Gate: mean meaning ≥ 4. Fluency and additions are reported.

**Sheet B — blind persona recognition (6-way).** For each sample source the six outputs are shuffled and labelled A–F. The reviewer, without looking at Cell 10, assigns each letter one of the six personas, using each persona once (a permutation). Accuracy = correct assignments / 30. Chance is 1 in 6 (17 %); the stretch gate is 50 %. The shuffle seed depends on the decoder, so the three runs have different keys.

In [26]:
# Cell 8 — Sheet A (meaning, fluency, additions) and Sheet B (blind persona recognition)

SAMPLE = [r for r in select("lora", {"validation"}) if r["source_id"] in SAMPLE_SOURCES]
print("SHEET A — score meaning (1-5, vs the English source), fluency (1-5), additions (none / harmless / material)\n")
for i, r in enumerate(SAMPLE, 1):
    print(f"A{i:02d} | {r['source_id']} | {NAME[r['persona']]}")
    print(f"      English: {r['source']}")
    print(f"      Output:  {show(r)}")
    print("      meaning __   fluency __   additions ______\n")

KEY = {}
shuffle = random.Random(f"{DECODER}-blind-{SEED}")
print("\nSHEET B — assign each letter one persona (each persona once per source)")
print("Personas: " + ", ".join(NAME[p] for p in PERSONAS))
for sid in SAMPLE_SOURCES:
    rows = [r for r in SAMPLE if r["source_id"] == sid]
    order = shuffle.sample(rows, len(rows))
    KEY[sid] = {letter: r["persona"] for letter, r in zip("ABCDEF", order)}
    print(f"\n=== {sid} English: {rows[0]['source']}")
    for letter, r in zip("ABCDEF", order):
        print(f"  {letter}. {r['output']}")

SHEET A — score meaning (1-5, vs the English source), fluency (1-5), additions (none / harmless / material)

A01 | s041 | Vijay (Deewar)
      English: We should celebrate the progress without ignoring the remaining work.
      Output:  Aaj bharosa hai, lekin khatam nahi. Phir kaam hoga.
      meaning __   fluency __   additions ______

A02 | s041 | Don
      English: We should celebrate the progress without ignoring the remaining work.
      Output:  Progress ka bharosa hai. Remaining work ko ek baar check karo. Two steps ahead.
      meaning __   fluency __   additions ______

A03 | s041 | Shahenshah
      English: We should celebrate the progress without ignoring the remaining work.
      Output:  Aaj bhi khatam nahi hoga. Aapki dastakar ka daftari se pehle apni tareef par baat karne ki koshish karo.
      meaning __   fluency __   additions ______

A04 | s041 | Anthony Gonsalves
      English: We should celebrate the progress without ignoring the remaining work.
      Output:  Prog

## Cell 9 — Blind test split (report only)

The kept checkpoint on the 60 test pairs, generated once after the checkpoint was chosen. Reported for the record; **not used to choose the decoder or the checkpoint.**

In [27]:
# Cell 9 — Test split summary (LoRA only), reported, not used for decisions

s = SUMMARY[("lora", "test")] = summary(select("lora", {"test"}))
print(f"test, LoRA: n={s['n']} | stop {s['stop']:.0%} | repetition {s['repetition']:.0%} | "
      f"Devanagari {s['devanagari']:.0%} | untranslated {s['untranslated']:.0%} | chrF {s['chrf']:.1f} | "
      f"persona overlap {s['persona_overlap']:.3f} | median tokens {s['median_tokens']:.0f}")

test, LoRA: n=60 | stop 100% | repetition 0% | Devanagari 0% | untranslated 0% | chrF 20.8 | persona overlap 0.147 | median tokens 27


## Cell 10 — The key for Sheet B, settings, and every record

The answer key for Sheet B, then the settings and every output (validation, out-of-domain, test, and the per-epoch probes) as JSON lines between markers, so all sheets and summaries can be rebuilt from the saved notebook.

In [28]:
# Cell 10 — Key, settings, records

print("KEY (Sheet B)")
for sid, k in KEY.items():
    print(f"  {sid}  " + "  ".join(f"{letter}={NAME[p]}" for letter, p in k.items()))

SETTINGS = {
    "versions": VERSIONS,
    "data": {"url": DATA, "train": len(TRAIN), "validation": len(VAL), "test": len(TEST), "ood": OOD_SOURCES,
             "sample_sources": SAMPLE_SOURCES, "probe_sources": PROBE_SOURCES},
    "prompt": {"head": HEAD, "description": " Persona: <description>", "tail": TAIL},
    "lora": {"r": LORA.r, "alpha": LORA.lora_alpha, "dropout": LORA.lora_dropout,
             "targets": TARGETS if isinstance(TARGETS, str) else sorted(TARGETS)},
    "precision": "4-bit NF4 weights, fp32 compute" if IS_GEMMA else "fp32",
    "training": {"max_epochs": MAX_EPOCHS, "patience": PATIENCE, "micro_batch": MICRO, "accumulation": ACCUM,
                 "lr": LR, "schedule": "linear decay over the maximum", "gradient_checkpointing": IS_GEMMA,
                 "seed": SEED, "kept_epoch": BEST_EPOCH, "seconds": TRAIN_SECONDS, "log": LOG,
                 "selection": "lowest validation loss"},
    "generation": {"do_sample": False, "max_new_tokens": MAX_NEW_TOKENS, "repetition_guard": None,
                   "stop_token_ids": STOPS},
    "summary": {f"{v} / {k}": s for (v, k), s in SUMMARY.items()},
    "automatic_gates": GATES,
}
print("\n----- SETTINGS -----")
print(json.dumps(SETTINGS, ensure_ascii=False, indent=1, default=str))
print("\n----- RECORDS (JSON lines) -----")
for r in RESULTS:
    print(json.dumps(r, ensure_ascii=False))
for r in PROBES:
    print(json.dumps({"variant": "probe", "epoch": r["epoch"], "id": r["id"], "output": r["output"],
                      "ended": r["ended"]}, ensure_ascii=False))
print("----- END RECORDS -----")

KEY (Sheet B)
  s041  A=Shahenshah  B=Vijay Dinanath Chauhan (Agneepath)  C=Don  D=Jai (Sholay)  E=Anthony Gonsalves  F=Vijay (Deewar)
  s043  A=Anthony Gonsalves  B=Don  C=Jai (Sholay)  D=Vijay Dinanath Chauhan (Agneepath)  E=Vijay (Deewar)  F=Shahenshah
  s045  A=Shahenshah  B=Vijay Dinanath Chauhan (Agneepath)  C=Don  D=Anthony Gonsalves  E=Vijay (Deewar)  F=Jai (Sholay)
  s047  A=Vijay Dinanath Chauhan (Agneepath)  B=Anthony Gonsalves  C=Shahenshah  D=Don  E=Vijay (Deewar)  F=Jai (Sholay)
  s049  A=Jai (Sholay)  B=Vijay (Deewar)  C=Anthony Gonsalves  D=Vijay Dinanath Chauhan (Agneepath)  E=Shahenshah  F=Don

----- SETTINGS -----
{
 "versions": {
  "decoder": "qwen-instruct+hinglish",
  "torch": "2.11.0+cu128",
  "transformers": "5.16.1",
  "peft": "0.20.0",
  "device": "Tesla T4",
  "revisions": {
   "Qwen/Qwen2.5-1.5B-Instruct": "989aa7980e4cf806f80c7fef2b1adb7bc71aa306",
   "sarthaksenapati/qwen1.5b-hinglish-sft-v2": "942fd0b2ebd2119d772f18952de6f8d372a6448b"
  },
  "adapter": {


## Cell 11 — Optional: save the adapter

Writes the kept LoRA to `/kaggle/working/translation-<decoder>-adapter`. Files in `/kaggle/working` are lost when an interactive session ends; download it only if this decoder is chosen. Nothing is uploaded.

In [29]:
# Cell 11 — Save the kept LoRA adapter locally on Kaggle

PATH = f"/kaggle/working/translation-{DECODER.replace('+', '-')}-adapter"
model.save_pretrained(PATH)
print(PATH)

/kaggle/working/translation-qwen-instruct-hinglish-adapter


---
## Next

Save the notebook (clear the token in Cell 0 first). Then compare this run with the other two decoders' notebooks on:

1. the automatic gates and diagnostics (Cell 7), and resources (memory, seconds per output, training time);
2. Sheet A (meaning ≥ 4 mean; fluency; additions) and Sheet B (blind persona recognition) on the same 30 pairs;
3. the decision: the decoder for the bridge is chosen on validation results, stopping, translation quality and resources. The adapter-merged run is diagnostic only. None of this shows the bridge works; that is Phase 4.